In [48]:
from sklearn.svm import SVR, LinearSVR
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error

---
## <u>Load Dataset</u>

In [14]:
db_data = datasets.load_diabetes(as_frame = True).frame
db_data.head()
# db_data.info() # no feature encoding needed 

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


---
## <u>Train Test Split</u> 

In [7]:
X = db_data.drop(columns = ["target"])
y = db_data["target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 42) 

---
## <u>Scaling</u>

In [27]:
# for our diabetes dataset, the X values are already scaled but y values are not
# the y column is just a 1d array and standard scaler demands a 2d array so first we convert to df

y_train = pd.DataFrame(y_train)
y_test = pd.DataFrame(y_test)


scaler = StandardScaler()

y_train_scaled = scaler.fit_transform(y_train).flatten() # we use .flatten() to again return the output at a 1d array only
y_test_scaled = scaler.transform(y_test).flatten() # when we fit data to the model, it demands tha labels should be an 1d array

# y_train_scaled = np.array(y_train_scaled)
# y_test_scaled = np.array(y_test_scaled)

---
# <u>Create and Train the model</u>

In [28]:
svr_model = SVR() # by default c = 1.0, kernel = "rbf" (gaussian) and epsilon = 0.1
svr_model.fit(X_train, y_train_scaled)

,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,tol,0.001
,C,1.0
,epsilon,0.1
,shrinking,True
,cache_size,200
,verbose,False
,max_iter,-1


---
## <u>Make Predictions</u>

In [40]:
y_pred_train_scaled = svr_model.predict(X_train)
y_pred_test_scaled = svr_model.predict(X_test)

---
## <u>Evaluate</u>

In [42]:
print("For baseline svr model :-\n")

print("For train data :-\n")

print("R - squared score: ", r2_score(y_train_scaled, y_pred_train_scaled))

print("For test data :-\n")

print("R - squared score: ", r2_score(y_test_scaled, y_pred_test_scaled))

For baseline svr model :-

For train data :-

R - squared score:  0.6596361676267714
For test data :-

R - squared score:  0.48844443151651906


---
## <u>Hyper parameter tuning</u>

In [47]:
steps = [("svr", SVR())]
pipeline = Pipeline(steps)

param_grid = {
    "svr__C" : [1, 2, 5, 10, 50, 100],
    "svr__epsilon" : [0.01, 0.1, 0.2, 0.3, 0.5],
    "svr__kernel" : ["rbf", "linear", "poly", "sigmoid"]
}

svr_model_cv = GridSearchCV(
    pipeline,
    param_grid,
    cv = 5,
    scoring = "r2"
)

svr_model_cv.fit(X_train, y_train_scaled)
y_pred_train_scaled = svr_model_cv.predict(X_train)
y_pred_test_scaled = svr_model_cv.predict(X_test)

print("best hyper parameters : ", svr_model_cv.best_params_)

print("For hypertuned svr model :-\n")

print("For train data :-\n")

print("R - squared score: ", r2_score(y_train_scaled, y_pred_train_scaled))

print("\nFor test data :-\n")

print("R - squared score: ", r2_score(y_test_scaled, y_pred_test_scaled))

best hyper parameters :  {'svr__C': 10, 'svr__epsilon': 0.1, 'svr__kernel': 'linear'}
For hypertuned svr model :-

For train data :-

R - squared score:  0.5151066486918875

For test data :-

R - squared score:  0.47444183250401084


---
## <u>Linear SVR</u>

- It is a simplified and similar version of SVR but with kernel as "linear" and less paramters 
- faster than normal SVR
- for large dataset we generally use Linear SVR but for small dataset we use SVR
- major difference is the loss functions of both

In [57]:
steps = [("lin_svr",  LinearSVR())]
pipeline = Pipeline(steps)

param_grid = {
    "lin_svr__C" : [1, 2, 5, 10, 50, 100],
    "lin_svr__epsilon" : [0.01, 0.1, 0.2, 0.3, 0.5],
    "lin_svr__max_iter" : [50000,100000]
}

lin_svr_model_cv = GridSearchCV(
    pipeline,
    param_grid,
    cv = 5,
    scoring = "r2"
)

lin_svr_model_cv.fit(X_train, y_train_scaled)
y_pred_train_scaled = lin_svr_model_cv.predict(X_train)
y_pred_test_scaled = lin_svr_model_cv.predict(X_test)

print("best hyper parameters : ", lin_svr_model_cv.best_params_)

print("For hypertuned svr model :-\n")

print("For train data :-\n")

print("R - squared score: ", r2_score(y_train_scaled, y_pred_train_scaled))

print("\nFor test data :-\n")

print("R - squared score: ", r2_score(y_test_scaled, y_pred_test_scaled))

best hyper parameters :  {'lin_svr__C': 10, 'lin_svr__epsilon': 0.1, 'lin_svr__max_iter': 100000}
For hypertuned svr model :-

For train data :-

R - squared score:  0.5154015063431221

For test data :-

R - squared score:  0.4748270892433566
